# Load the BC export into Lakehouse tables

Run in a Microsoft Fabric notebook attached to the lakehouse (e.g. `bc_reporting`).

1. Upload the folder `data/bc_export/deltas` (created by `make bc-export`) to **Files/** so you have `Files/deltas/Customer-18/`, `Files/deltas/CustLedgerEntry-21/`, ...
2. Run all cells. Each Delta folder becomes a managed table named `<Table>_<Id>` (e.g. `CustLedgerEntry_21`), the name the views in `sql/ax_compat/` expect.
3. Then open the lakehouse **SQL analytics endpoint** and run `sql/ax_compat/00..06` in order.

All data is synthetic (two invented companies plus an unmapped sandbox).

In [ ]:
BASE = "Files/deltas"

loaded = []
for entry in notebookutils.fs.ls(BASE):
    folder = entry.name.rstrip("/")
    table = folder.replace("-", "_")
    df = spark.read.format("delta").load(f"{BASE}/{folder}")
    (
        df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        # BC column names contain '-' and '$'; column mapping keeps them as-is in Delta.
        .option("delta.columnMapping.mode", "name")
        .saveAsTable(table)
    )
    loaded.append((table, df.count()))

display(spark.createDataFrame(loaded, ["table", "rows"]))

In [ ]:
# BC company name -> AX DATAAREAID. Adding a company is a row here, not a code change.
company_map = spark.createDataFrame(
    [("Contoso Global Ltd.", "cgl"), ("Contoso India Pvt. Ltd.", "cin")],
    ["CompanyName", "DATAAREAID"],
)
company_map.write.mode("overwrite").saveAsTable("company_map")
display(company_map)

In [ ]:
# Change data feed on the source tables lets materialized lake views and downstream loads
# refresh incrementally instead of re-reading everything.
for table, _ in loaded:
    spark.sql(f"ALTER TABLE {table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("CDF enabled on", [t for t, _ in loaded])

In [ ]:
%%sql
-- Quick check in Spark SQL: rows per company in the customer ledger
SELECT `$Company` AS company, COUNT(*) AS entries, SUM(CASE WHEN `Open-36` THEN 1 ELSE 0 END) AS open_entries
FROM CustLedgerEntry_21
GROUP BY `$Company`
ORDER BY company